<a href="https://colab.research.google.com/github/JPMwangi/kenya-econ-pdf-extractor/blob/main/knbs_pdf_extraction_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# KNBS Leading Economic Indicators (August 2026) — PDF Table Extraction Experiment

**Objective:** Evaluate programmatic table extraction from the Kenya National Bureau of Statistics (KNBS) *Leading Economic Indicators (August 2026)* PDF report using `pdfplumber` and `pandas`, document structural roadblocks, and establish evidence for pivoting to an AI-native natural language extraction workflow (NotebookLM).

**Source File:** `/content/Kenya-Leading-Economic-Indicators-August-2026.pdf`

## Step 1: Environment Setup
Install required PDF parsing and data manipulation libraries.

In [ ]:
!pip install pdfplumber pandas matplotlib seaborn

## Step 2: Initial Attempt — Direct Page Table Extraction
Attempted to extract **Table 3: Interest Rates** from physical page 9 using `pdfplumber`'s default grid-based table extraction.

In [ ]:
import pdfplumber
import pandas as pd

pdf_path = "/content/Kenya-Leading-Economic-Indicators-August-2026.pdf"

def extract_table_from_page(pdf_file, page_num, table_settings=None):
    with pdfplumber.open(pdf_file) as pdf:
        page = pdf.pages[page_num]
        tables = page.extract_tables(table_settings or {})
        if tables:
            return pd.DataFrame(tables[0])
        else:
            print(f"No tables found on page {page_num + 1}")
            return None

target_page_index = 8  # Page 9 (0-based index)
df_raw = extract_table_from_page(pdf_path, target_page_index)

No tables found on page 9


## Step 3: Diagnostic Scan — Mapping Embedded Grid Tables Across All 51 Pages
Because page 9 returned no grid tables, we ran a full document scan to identify which physical pages contained detectable table grids.

In [ ]:
with pdfplumber.open(pdf_path) as pdf:
    for i, page in enumerate(pdf.pages):
        tables = page.extract_tables()
        if tables:
            print(f"✅ Page {i+1} has {len(tables)} table(s)")
        else:
            print(f"❌ Page {i+1} has no tables")

❌ Page 1 has no tables
❌ Page 2 has no tables
❌ Page 3 has no tables
❌ Page 4 has no tables
❌ Page 5 has no tables
❌ Page 6 has no tables
❌ Page 7 has no tables
❌ Page 8 has no tables
❌ Page 9 has no tables
❌ Page 10 has no tables
❌ Page 11 has no tables
❌ Page 12 has no tables
❌ Page 13 has no tables
❌ Page 14 has no tables
❌ Page 15 has no tables
✅ Page 16 has 1 table(s)
❌ Page 17 has no tables
❌ Page 18 has no tables
❌ Page 19 has no tables
❌ Page 20 has no tables
❌ Page 21 has no tables
❌ Page 22 has no tables
✅ Page 23 has 1 table(s)
❌ Page 24 has no tables
❌ Page 25 has no tables
❌ Page 26 has no tables
❌ Page 27 has no tables
❌ Page 28 has no tables
✅ Page 29 has 1 table(s)
❌ Page 30 has no tables
❌ Page 31 has no tables
❌ Page 32 has no tables
✅ Page 33 has 1 table(s)
❌ Page 34 has no tables
❌ Page 35 has no tables
❌ Page 36 has no tables
❌ Page 37 has no tables
❌ Page 38 has no tables
✅ Page 39 has 1 table(s)
❌ Page 40 has no tables
✅ Page 41 has 1 table(s)
❌ Page 42 has no ta

## Step 4: Discovering the List of Tables Index
Scanned the front matter to map table titles to their logical report page numbers.

In [ ]:
with pdfplumber.open(pdf_path) as pdf:
    print("--- Scanning for List of Tables ---")
    for i in range(len(pdf.pages)):
        text = pdf.pages[i].extract_text()
        if text and ("Table" in text and ("Inflation" in text or "Consumer" in text or "Table 1" in text)):
            print(f"\n--- Found potential Table List on PDF Page {i+1} ---")
            print(text[:1500])
            break

--- Scanning for List of Tables ---

--- Found potential Table List on PDF Page 2 ---
List of Tables Page Number
Table 1(a): Consumer Price Indices ................................................................................................................... 6
Table 1(b): Inflation Rates ................................................................................................................................. 7
Table 2: Mean Monthly Foreign Exchange Rates of Kenya Shilling against Selected Major Currencies ......... 8
Table 3: Interest Rates (%) ................................................................................................................................ 9
Table 4: Nairobi Securities Exchange – Monthly Equity Market Performance ............................................... 10
Table 5(a): Money Supply ............................................................................................................................... 11
Table 5(b): Gross Foreign 

## Step 5: Inspecting Raw Text Layout & Uncovering the Root Cause
When falling back to raw text extraction on physical page 9 (which holds logical report Page 6, **Table 1(a): Consumer Price Indices**), we discovered two major issues:
1. **Page Offset Mismatch:** Physical PDF Page 9 corresponds to report Page 6 due to introductory front matter.
2. **Wide Character Spacing Encoding:** The PDF encodes tabular characters with spaces between individual glyphs (`P e r i o d / G r o u p`), breaking automated grid detection and multi-column alignment.

In [ ]:
target_pdf_page = 9
page_index = target_pdf_page - 1

with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[page_index]
    tables = page.extract_tables()
    if not tables:
        print(f"No table grid found on page {target_pdf_page}. Checking text layout...")
        print(page.extract_text())

No table grid found on page 9. Checking text layout...
Table 1(a): Consumer Price Indices
Pa g e |6
Leading Economic Indicators August 2026
N a i r o b i
P e r i o d / G r o u p L o w e r M i d d l e U p p e r
I n c o m e I n c o m e I n c o m e
2 0 2 4
S e p t e m b e r 1 4 1 .3 4 1 2 9 .1 2 1 2 7 .4 7
O c t o b e r 1 4 1 .4 8 1 2 9 .8 6 1 2 7 .7 5
N o v e m b e r 1 4 1 .8 9 1 3 0 .3 7 1 2 8 .0 9
D e c e m b e r 1 4 3 .2 6 1 3 0 .8 3 1 2 8 .4 7
2 0 2 5
J a n u a r y 1 4 4 .5 5 1 3 1 .7 9 1 2 8 .9 6
F e b r u a r y 1 4 4 .8 9 1 3 2 .4 2 1 2 9 .7 6
M a r c h 1 4 5 .4 2 1 3 2 .8 6 1 3 0 .4 4
A p r i l 1 4 5 .6 7 1 3 3 .2 3 1 3 0 .7 2
M a y 1 4 6 .1 5 1 3 4 .5 7 1 3 1 .1 9
J u n e 1 4 6 .9 9 1 3 5 .1 8 1 3 1 .7 1
J u l y 1 4 7 .2 2 1 3 5 .5 3 1 3 2 .6 3
A u g u s t 1 4 7 .9 1 1 3 5 .6 7 1 3 2 .9 0
S e p t e m b e r 1 4 8 .1 2 1 3 6 .0 7 1 3 3 .2 0
O c t o b e r 1 4 8 .4 3 1 3 6 .2 7 1 3 3 .5 8
N o v e m b e r 1 4 8 .5 6 1 3 6 .5 7 1 3 4 .1 8
D e c e m b e r 1 4 9 .8 5 1 3 7 .6 5 1 3 5 .5 

## Step 6: String-Search Table Locator Test
Testing string-matching to find the first occurrence of `'Table 1(a): Consumer Price Indices'`, which matched the Table of Contents entry on Page 2 rather than the data page itself.

In [ ]:
target_table_title = "Table 1(a): Consumer Price Indices"

with pdfplumber.open(pdf_path) as pdf:
    for i, page in enumerate(pdf.pages):
        text = page.extract_text()
        if text and target_table_title in text:
            print(f"🎯 Found '{target_table_title}' on physical PDF Page {i + 1}!")
            break

🎯 Found 'Table 1(a): Consumer Price Indices' on physical PDF Page 2!


## Conclusion & Operational Pivot

As demonstrated by the outputs above:
1. **Fragile Grid Detection:** Over 78% of pages (40 of 51) failed standard grid extraction.
2. **Corrupted Column Serialization:** Notice in Step 5 how the right-hand columns (`Nairobi Combined`, `Rest of Urban Areas`, `Kenya CPI`) and footnotes interleaved below the left-hand columns during text extraction, alongside character-level spacing (`1 4 1 .3 4`).
3. **Decision:** Rather than maintaining custom coordinate-clustering and regex de-spacing pipelines for every monthly KNBS release, this notebook is archived as empirical evidence justifying our transition to **NotebookLM** for natural-language semantic table extraction.